# 5. MD stability of the docked complex

Docking scores a single **rigid** pose. This notebook asks the next question:
once the protein and ligand can move and real water is present, **does the pose survive?**

We run explicit-solvent molecular dynamics with OpenMM on the best zinc-coordinating
pose from notebook 04 and track four things over time:

| Metric | What it tells us |
|---|---|
| Ligand RMSD | how far the ligand has drifted from its docked pose |
| Backbone RMSD | whether the protein itself stayed folded (a sanity check) |
| Sulfonamide N &ndash; Zn distance | whether the defining zinc contact is kept |
| Ligand&ndash;protein hydrogen bonds | how many specific contacts persist |

## Read this before trusting the output

**Zinc is modelled as a non-bonded ion.** Amber represents Zn(2+) as a point charge
plus a Lennard-Jones term. Real sulfonamide&ndash;zinc binding has substantial covalent
character, so this force field *systematically under-binds that contact*. If the
N&ndash;Zn distance grows during the run, that is a known limitation of the model, not
evidence that the docked pose is wrong. Treating it properly needs a bonded zinc
model (ZAFF), QM/MM, or a metal-aware polarisable force field.

**This is CPU-bound.** Measured throughput for this system (37,800 atoms, 4 fs
timestep with heavy hydrogens) is about **2 ns/day on 4 CPU cores**. The default
below is a short run that exercises the whole pipeline and produces real plots in
roughly 15&ndash;30 minutes. **Conclusions about stability need 10 ns or more, which
in practice means a CUDA GPU** (the same system runs far faster there). A 20 ps run
tells you the complex does not explode; it does not tell you the pose is stable.

## Requirements

Not pip-installable &mdash; use conda:

```bash
conda create -n ca-md -c conda-forge python=3.11 openmm openmmforcefields \
    openff-toolkit pdbfixer mdtraj rdkit matplotlib
conda activate ca-md
```

Run this notebook from the **project root** so the `artifacts/` paths resolve.

In [ ]:
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import openmm

if not os.path.exists("artifacts"):
    os.chdir("..")          # tolerate being launched from notebook/
sys.path.insert(0, os.getcwd())

from src.analysis import md

# --- Configuration -------------------------------------------------------
PDB_ID = "3HS4"                                            # CA II, acetazolamide complex
LIGAND_SDF = "artifacts/md/3HS4_acetazolamide_pose.sdf"    # docked pose from notebook 04
LIGAND_RESNAME = "UNK"                                     # name OpenMM gives the ligand
SIM_TIME_NS = 0.02                                         # raise to 10.0 on a GPU
OUT_DIR = os.path.join("artifacts", "md", f"{PDB_ID}_run")
os.makedirs(OUT_DIR, exist_ok=True)

platforms = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
PLATFORM = next(p for p in ("CUDA", "OpenCL", "CPU") if p in platforms)
print("OpenMM", openmm.__version__, "| available:", platforms, "| using:", PLATFORM)

measured_ns_per_day = {"CUDA": 120.0, "OpenCL": 40.0, "CPU": 2.0}[PLATFORM]
minutes = SIM_TIME_NS / measured_ns_per_day * 24 * 60
print(f"\n{SIM_TIME_NS} ns at roughly {measured_ns_per_day:g} ns/day on {PLATFORM}"
      f" -> about {minutes:.0f} minutes of production")
if PLATFORM == "CPU" and SIM_TIME_NS >= 1:
    print("WARNING: that is over 12 hours on CPU. Lower SIM_TIME_NS or use a GPU.")

## Step 1 &mdash; Build the solvated complex

The receptor is one protein chain plus its catalytic zinc: waters, co-crystallised
ligands and the other chains are dropped. PDBFixer rebuilds missing atoms and adds
hydrogens at pH 7.4; the ligand keeps its **docked coordinates** and is typed with
GAFF 2.11; then the whole thing is solvated in TIP3P with 0.15 M salt.

Note which pose we start from &mdash; the zinc-coordinating one, which AutoDock4Zn
ranked *second* rather than first.

In [ ]:
from rdkit import Chem

receptor_pdb, chain = md.extract_receptor(
    os.path.join("artifacts", "structures", f"{PDB_ID.lower()}.pdb"),
    out_path=os.path.join(OUT_DIR, f"{PDB_ID}_receptor.pdb"),
)
print(f"receptor: chain {chain} -> {receptor_pdb}")

pose = Chem.SDMolSupplier(LIGAND_SDF, removeHs=False)[0]
print("starting pose:", {name: pose.GetProp(name) for name in pose.GetPropNames()})

In [ ]:
modeller, generator, ligand = md.build_complex(receptor_pdb, LIGAND_SDF)
print(f"ligand: {ligand.n_atoms} atoms, formal charge {ligand.total_charge}")
print(f"solvated system: {modeller.topology.getNumAtoms():,} atoms")

residues = [r.name for r in modeller.topology.residues()]
print("waters:", residues.count("HOH"), "| zinc present:", "ZN" in residues)
assert LIGAND_RESNAME in residues, f"expected a {LIGAND_RESNAME} residue, got {set(residues)}"

## Step 2 &mdash; Minimise, equilibrate, run

Minimisation relieves clashes left by docking into a rigid receptor, 10 ps of NVT
equilibration brings the system to 300 K, then production runs at constant pressure
so the box density can relax.

The **minimised** structure is written out and used as the reference frame for every
RMSD below, so an RMSD near 0 means "still in the docked pose".

In [ ]:
run = md.run_simulation(
    modeller, generator, ligand,
    nanoseconds=SIM_TIME_NS,
    out_dir=OUT_DIR,
    platform_name=PLATFORM,
    progress=lambda message: print(" ", message, flush=True),
)
print()
print(json.dumps(run, indent=2))

## Step 3 &mdash; Analyse the trajectory

Frames are superposed on the protein backbone first, so the ligand RMSD measures
movement *relative to the binding site* rather than tumbling of the whole protein.

In [ ]:
metrics = md.analyse_trajectory(run["topology_pdb"], run["trajectory_dcd"],
                                ligand_resname=LIGAND_RESNAME)
summary = md.summarise(metrics)

print(f"{metrics['n_frames']} frames | "
      f"{len(metrics['ligand_atoms'])} ligand heavy atoms | "
      f"sulfonamide N indices: {metrics['sulfonamide_nitrogens']}")
print()
print(json.dumps(summary, indent=2))

In [ ]:
time_ns = np.asarray(metrics["time_ps"]) / 1000.0
fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
fig.suptitle(f"{PDB_ID} + acetazolamide, {SIM_TIME_NS} ns explicit solvent ({run['platform']})",
             fontweight="bold")

axes[0, 0].plot(time_ns, metrics["ligand_rmsd"], color="#2563eb")
axes[0, 0].axhline(2.0, ls="--", c="#64748b", lw=1, label="2 A: pose retained")
axes[0, 0].set(xlabel="time (ns)", ylabel="RMSD (A)", title="Ligand drift from the docked pose")
axes[0, 0].legend(fontsize=8)

axes[0, 1].plot(time_ns, metrics["backbone_rmsd"], color="#7c3aed")
axes[0, 1].set(xlabel="time (ns)", ylabel="RMSD (A)", title="Protein backbone (fold sanity check)")

if metrics["zinc_distance"] is not None:
    axes[1, 0].plot(time_ns, metrics["zinc_distance"], color="#dc2626")
    axes[1, 0].axhline(2.5, ls="--", c="#64748b", lw=1, label="2.5 A: coordination cutoff")
    axes[1, 0].legend(fontsize=8)
axes[1, 0].set(xlabel="time (ns)", ylabel="distance (A)", title="Sulfonamide N - Zn")

axes[1, 1].step(time_ns, metrics["hydrogen_bonds"], where="mid", color="#059669")
axes[1, 1].set(xlabel="time (ns)", ylabel="count", title="Ligand-protein hydrogen bonds")

for ax in axes.flat:
    ax.grid(alpha=0.25)

figure_path = os.path.join(OUT_DIR, "stability.png")
fig.savefig(figure_path, dpi=150)
print("saved", figure_path)

## Step 4 &mdash; What the numbers mean

Read the summary above against the conventional thresholds for judging a docked
pose in MD:

| Metric | Pose looks retained | Pose looks lost |
|---|---|---|
| Ligand RMSD (mean, post-equilibration) | below ~2 &#8491; | above ~3 &#8491;, or rising steadily |
| Backbone RMSD | below ~2.5 &#8491; and flat | rising (the protein itself is unstable) |
| N&ndash;Zn distance | near 2 &#8491;, `zinc_contact_fraction` near 1 | drifting past 3 &#8491; |
| Hydrogen bonds | roughly constant | decaying to 0 |

### What the default 20 ps run actually produced

Running exactly this notebook on 4 CPU cores (3HS4 + acetazolamide, 91 frames):

```
backbone_rmsd_mean     0.66   protein stayed folded - the setup is sound
ligand_rmsd_mean       2.85   ligand drifted from the docked pose
hydrogen_bonds_mean    4.44   still making contacts in the pocket
zinc_distance_mean     3.38   coordination not retained
zinc_contact_fraction  0.00
```

The temperature held at 300 K and the box density relaxed to 1.02 g/mL, so the
simulation itself is healthy. But note where the zinc distance starts: the docked
pose had the sulfonamide nitrogen **1.9 &#8491;** from the zinc, and it is already
**3.2 &#8491;** in the first frame &mdash; *energy minimisation alone pushed the ligand
off the metal*, before any dynamics ran. That is the non-bonded zinc model (with a
neutral sulfonamide) failing to hold a coordination bond, exactly as warned at the
top. It is the clearest possible illustration of why this particular force field
cannot be used to judge sulfonamide&ndash;zinc binding, and why the ligand RMSD then
climbs. The ligand does stay in the pocket, which the retained hydrogen bonds show.

The honest conclusion from this run is therefore **about the method, not the
molecule**: acetazolamide is a potent CA II inhibitor in reality, and nothing here
contradicts that. To study this contact properly you need a bonded zinc model
(ZAFF), the deprotonated sulfonamide, or QM/MM.

**Interpret a short run narrowly.** With the default `SIM_TIME_NS` the run is long
enough to show the system is physically sound (energy minimises, temperature and
density settle, nothing blows up) and that the analysis works end to end. It is
*far* too short to call a pose stable &mdash; ligands routinely sit still for tens of
picoseconds and then leave. Re-run with `SIM_TIME_NS = 10` on a GPU before drawing
any conclusion, and ideally run **three independent replicas** with different
velocity seeds, since one trajectory is a single sample, not a measurement.

**Expect the zinc contact to loosen**, for the force-field reason given at the top.
If `zinc_contact_fraction` is low while the ligand RMSD stays small, the most likely
reading is that the ligand is still in the pocket in roughly the right orientation,
but the non-bonded zinc model cannot hold the coordination distance.

### Limitations of this setup

- Non-bonded zinc, as described above &mdash; the single biggest approximation here.
- One replica, and a default run far shorter than a stability claim requires.
- **No binding free energy.** MM-PBSA/MM-GBSA numbers from one short trajectory are
  not reliable, so none are reported. Use alchemical methods (FEP/TI) for affinities.
- The sulfonamide is modelled **neutral**, whereas it coordinates zinc as the anion.
  Another reason this setup under-represents the real interaction.
- Hydrogen mass repartitioning (3 amu) buys the 4 fs timestep. It is standard and
  does not change equilibrium structure, but it distorts fast dynamics &mdash; do not
  read kinetics off this trajectory.